In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os
import time
from datetime import datetime

# =============================================================================
# CONFIGURAZIONE & COSTANTI
# =============================================================================
class Config:
    WIDTH, HEIGHT = 1280, 720
    APP_NAME = "Virtual Painter Pro - Thesis Edition"
    OUTPUT_FOLDER = "Artworks_Gallery"
    
    # Colori (BGR)
    COLORS = {
        "NERO":    (0, 0, 0),       
        "BIANCO":  (255, 255, 255), 
        "ROSSO":   (0, 0, 255),
        "VERDE":   (0, 255, 0),
        "BLU":     (255, 0, 0),
        "GIALLO":  (0, 255, 255),
        "VIOLA":   (255, 0, 255),
        "ARANCIO": (0, 69, 255),
        "CIANO":   (255, 255, 0)
    }
    
    # UI Layout
    SIDEBAR_WIDTH = 120
    BTN_HEIGHT = 55      # Leggermente ridotto per far stare tutto
    BTN_MARGIN = 10
    
    # Smoothing
    SMOOTHING_FACTOR = 0.2 
    
    # Undo History Limit
    MAX_HISTORY = 10

# =============================================================================
# MODULO RILEVAMENTO MANO (WRAPPER)
# =============================================================================
class HandDetector:
    def __init__(self, mode=False, max_hands=1, detection_con=0.8, track_con=0.5):
        self.mode = mode
        self.max_hands = max_hands
        self.detection_con = detection_con
        self.track_con = track_con
        
        self.mpHands = mp.solutions.hands
        self.hands = self.mpHands.Hands(
            static_image_mode=self.mode,
            max_num_hands=self.max_hands,
            min_detection_confidence=self.detection_con,
            min_tracking_confidence=self.track_con
        )
        self.mpDraw = mp.solutions.drawing_utils
        self.tipIds = [4, 8, 12, 16, 20] 

    def find_hands(self, img, draw=True):
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(imgRGB)
        if self.results.multi_hand_landmarks and draw:
            for handLms in self.results.multi_hand_landmarks:
                self.mpDraw.draw_landmarks(img, handLms, self.mpHands.HAND_CONNECTIONS)
        return img

    def find_position(self, img):
        self.lmList = []
        if self.results.multi_hand_landmarks:
            myHand = self.results.multi_hand_landmarks[0]
            for id, lm in enumerate(myHand.landmark):
                h, w, c = img.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                self.lmList.append([id, cx, cy])
        return self.lmList

    def fingers_up(self):
        fingers = []
        if len(self.lmList) == 0: return []

        if self.lmList[self.tipIds[0]][1] < self.lmList[self.tipIds[0] - 1][1]:
            fingers.append(1)
        else:
            fingers.append(0)

        for id in range(1, 5):
            if self.lmList[self.tipIds[id]][2] < self.lmList[self.tipIds[id] - 2][2]:
                fingers.append(1)
            else:
                fingers.append(0)
        return fingers

# =============================================================================
# MOTORE GRAFICO & UI
# =============================================================================
class PainterEngine:
    def __init__(self):
        # Canvas Setup
        self.img_canvas = np.zeros((Config.HEIGHT, Config.WIDTH, 3), np.uint8)
        self.undo_stack = [] 
        self.redo_stack = []
        
        # Stato Pennello (Dimensione fissa ora)
        self.brush_color = Config.COLORS["ROSSO"]
        self.brush_size = 15     # Dimensione standard pennello
        self.eraser_size = 60    # Dimensione standard gomma
        self.is_eraser = False
        
        # Coordinate Smoothing
        self.xp, self.yp = 0, 0 
        self.cx, self.cy = 0, 0 
        
        # Stato UI
        self.active_tool = "ROSSO"
        self.feedback_msg = ""
        self.feedback_timer = 0
        
        # Animazione Save
        self.save_anim_alpha = 0
        self.last_saved_preview = None

        # --- DEFINIZIONE LISTE BOTTONI SEPARATE ---
        
        # 1. Lista Colori (Dall'alto verso il basso)
        self.color_buttons = [
            {"name": "ROSSO",   "col": Config.COLORS["ROSSO"],   "type": "COLOR"},
            {"name": "VERDE",   "col": Config.COLORS["VERDE"],   "type": "COLOR"},
            {"name": "BLU",     "col": Config.COLORS["BLU"],     "type": "COLOR"},
            {"name": "GIALLO",  "col": Config.COLORS["GIALLO"],  "type": "COLOR"},
            {"name": "VIOLA",   "col": Config.COLORS["VIOLA"],   "type": "COLOR"},
            {"name": "CIANO",   "col": Config.COLORS["CIANO"],   "type": "COLOR"},
        ]
        
        # 2. Lista Utility (Dal basso verso l'alto: Clear è l'ultimo in basso)
        self.utility_buttons = [
            {"name": "CLEAR",   "col": (50, 50, 200),            "type": "ACTION"}, # Fondo
            {"name": "GOMMA",   "col": (50, 50, 50),             "type": "TOOL"},   # Sopra Clear
            {"name": "UNDO",    "col": (100, 100, 100),          "type": "ACTION"}, # Sopra Gomma
        ]
        
        if not os.path.exists(Config.OUTPUT_FOLDER):
            os.makedirs(Config.OUTPUT_FOLDER)

    def save_state_for_undo(self):
        if len(self.undo_stack) >= Config.MAX_HISTORY:
            self.undo_stack.pop(0) 
        self.undo_stack.append(self.img_canvas.copy())
        self.redo_stack.clear() 

    def perform_undo(self):
        if self.undo_stack:
            self.redo_stack.append(self.img_canvas.copy()) 
            self.img_canvas = self.undo_stack.pop()
            self.set_feedback("Undo")

    def draw_ui(self, img):
        # 1. Sidebar Background
        cv2.rectangle(img, (0, 0), (Config.SIDEBAR_WIDTH, Config.HEIGHT), (30, 30, 30), -1)
        cv2.line(img, (Config.SIDEBAR_WIDTH, 0), (Config.SIDEBAR_WIDTH, Config.HEIGHT), (80, 80, 80), 2)
        
        # 2. Disegna Colori (In alto)
        for i, btn in enumerate(self.color_buttons):
            y_pos = 20 + i * (Config.BTN_HEIGHT + Config.BTN_MARGIN)
            btn["rect"] = (10, y_pos, Config.SIDEBAR_WIDTH - 20, Config.BTN_HEIGHT) 
            
            # Highlight
            if btn["name"] == self.active_tool:
                cv2.rectangle(img, (5, y_pos-2), (Config.SIDEBAR_WIDTH-5, y_pos+Config.BTN_HEIGHT+2), (255, 255, 255), 2)
                
            cv2.rectangle(img, (10, y_pos), (Config.SIDEBAR_WIDTH - 10, y_pos + Config.BTN_HEIGHT), btn["col"], -1)
            cv2.putText(img, btn["name"], (20, y_pos + 40), cv2.FONT_HERSHEY_PLAIN, 1.1, (255, 255, 255), 2)

        # 3. Disegna Utility (In basso - Ancorati al fondo)
        # L'indice 0 è il più basso (CLEAR), 1 è sopra (GOMMA), ecc.
        for i, btn in enumerate(self.utility_buttons):
            # Calcolo posizione partendo da HEIGHT
            y_pos = Config.HEIGHT - ((i + 1) * (Config.BTN_HEIGHT + Config.BTN_MARGIN)) - 10
            btn["rect"] = (10, y_pos, Config.SIDEBAR_WIDTH - 20, Config.BTN_HEIGHT)
            
            # Highlight solo per GOMMA (Undo e Clear sono azioni istantanee)
            if btn["name"] == self.active_tool:
                cv2.rectangle(img, (5, y_pos-2), (Config.SIDEBAR_WIDTH-5, y_pos+Config.BTN_HEIGHT+2), (255, 255, 255), 2)

            cv2.rectangle(img, (10, y_pos), (Config.SIDEBAR_WIDTH - 10, y_pos + Config.BTN_HEIGHT), btn["col"], -1)
            cv2.putText(img, btn["name"], (20, y_pos + 40), cv2.FONT_HERSHEY_PLAIN, 1.1, (255, 255, 255), 2)

        # 4. Bottone Salva (Alto Destra)
        self.save_btn_rect = (Config.WIDTH - 110, 10, 100, 50)
        cv2.rectangle(img, (self.save_btn_rect[0], self.save_btn_rect[1]), 
                      (self.save_btn_rect[0]+self.save_btn_rect[2], self.save_btn_rect[1]+self.save_btn_rect[3]), (50, 200, 50), -1)
        cv2.putText(img, "SALVA", (Config.WIDTH - 100, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    def update_feedback(self, img):
        if self.save_anim_alpha > 0:
            overlay = img.copy()
            cv2.rectangle(overlay, (0,0), (Config.WIDTH, Config.HEIGHT), (255,255,255), -1)
            cv2.addWeighted(overlay, self.save_anim_alpha/255.0, img, 1 - self.save_anim_alpha/255.0, 0, img)
            self.save_anim_alpha -= 20

        if self.feedback_timer > 0:
            cv2.putText(img, self.feedback_msg, (Config.WIDTH//2 - 100, Config.HEIGHT//2), 
                        cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 255), 3)
            self.feedback_timer -= 1
            
        if self.last_saved_preview is not None and self.feedback_timer > 0:
             h, w, _ = self.last_saved_preview.shape
             y_off = Config.HEIGHT - h - 10
             x_off = Config.WIDTH - w - 10
             cv2.rectangle(img, (x_off-2, y_off-2), (Config.WIDTH-8, Config.HEIGHT-8), (255,255,255), 2)
             img[y_off:y_off+h, x_off:x_off+w] = self.last_saved_preview

    def set_feedback(self, msg, duration=30):
        self.feedback_msg = msg
        self.feedback_timer = duration

    def save_artwork(self):
        filename = f"{Config.OUTPUT_FOLDER}/Art_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        cv2.imwrite(filename, self.img_canvas)
        self.save_anim_alpha = 255
        self.last_saved_preview = cv2.resize(self.img_canvas, (160, 90))
        self.set_feedback("Salvato!", 60)

# =============================================================================
# MAIN APP LOOP
# =============================================================================
def main():
    cap = cv2.VideoCapture(0)
    cap.set(3, Config.WIDTH)
    cap.set(4, Config.HEIGHT)
    
    detector = HandDetector(max_hands=1)
    engine = PainterEngine()
    
    pTime = 0
    drawing_mode_active = False 

    while True:
        success, frame = cap.read()
        if not success: break
        
        frame = cv2.flip(frame, 1)
        frame = detector.find_hands(frame)
        lmList = detector.find_position(frame)
        
        engine.draw_ui(frame)
        
        if len(lmList) != 0:
            x1, y1 = lmList[8][1:]  
            x2, y2 = lmList[12][1:] 
            
            fingers = detector.fingers_up()
            
            # --- SELEZIONE (Indice + Medio) ---
            if fingers[1] and fingers[2]:
                engine.xp, engine.yp = 0, 0 
                drawing_mode_active = False
                
                # Cursore
                cv2.rectangle(frame, (x1, y1-25), (x2, y2+25), engine.brush_color, -1)
                
                # CHECK SIDEBAR (Unisci le liste per il controllo collisioni)
                if x1 < Config.SIDEBAR_WIDTH:
                    # Controlla tutti i bottoni (Colori + Utility)
                    all_buttons = engine.color_buttons + engine.utility_buttons
                    
                    for btn in all_buttons:
                        bx, by, bw, bh = btn["rect"]
                        if bx < x1 < bx+bw and by < y1 < by+bh:
                            if btn["type"] == "COLOR":
                                engine.active_tool = btn["name"]
                                engine.brush_color = btn["col"]
                                engine.is_eraser = False
                            elif btn["type"] == "TOOL": 
                                engine.active_tool = btn["name"]
                                engine.is_eraser = True
                            elif btn["type"] == "ACTION":
                                if btn["name"] == "CLEAR":
                                    engine.save_state_for_undo() 
                                    engine.img_canvas[:] = 0
                                    engine.set_feedback("Pulito")
                                elif btn["name"] == "UNDO":
                                    if engine.feedback_timer == 0: 
                                        engine.perform_undo()

                # CHECK TASTO SALVA
                sbx, sby, sbw, sbh = engine.save_btn_rect
                if sbx < x1 < sbx+sbw and sby < y1 < sby+sbh:
                    if engine.feedback_timer == 0: 
                        engine.save_artwork()

            # --- DISEGNO (Solo Indice) ---
            elif fingers[1] and not fingers[2]:
                
                # Safe Zone Check
                safe_draw = (x1 > Config.SIDEBAR_WIDTH) and not (
                    x1 > engine.save_btn_rect[0] and y1 < engine.save_btn_rect[1] + engine.save_btn_rect[3]
                )

                if safe_draw:
                    if not drawing_mode_active:
                        engine.save_state_for_undo()
                        drawing_mode_active = True
                        engine.xp, engine.yp = x1, y1
                        engine.cx, engine.cy = x1, y1

                    # Smoothing
                    engine.cx = int(engine.cx * (1 - Config.SMOOTHING_FACTOR) + x1 * Config.SMOOTHING_FACTOR)
                    engine.cy = int(engine.cy * (1 - Config.SMOOTHING_FACTOR) + y1 * Config.SMOOTHING_FACTOR)
                    
                    cv2.circle(frame, (engine.cx, engine.cy), 15, engine.brush_color if not engine.is_eraser else (0,0,0), -1)
                    
                    thickness = engine.eraser_size if engine.is_eraser else int(engine.brush_size)
                    col = engine.brush_color if not engine.is_eraser else (0, 0, 0)
                    
                    if engine.xp != 0:
                        cv2.line(engine.img_canvas, (engine.xp, engine.yp), (engine.cx, engine.cy), col, thickness)
                        cv2.line(frame, (engine.xp, engine.yp), (engine.cx, engine.cy), col, thickness)
                    
                    engine.xp, engine.yp = engine.cx, engine.cy
                else:
                    engine.xp, engine.yp = 0, 0
            
            else:
                engine.xp, engine.yp = 0, 0
                drawing_mode_active = False

        imgGray = cv2.cvtColor(engine.img_canvas, cv2.COLOR_BGR2GRAY)
        _, imgInv = cv2.threshold(imgGray, 10, 255, cv2.THRESH_BINARY_INV)
        imgInv = cv2.cvtColor(imgInv, cv2.COLOR_GRAY2BGR)
        
        frame = cv2.bitwise_and(frame, imgInv)
        frame = cv2.bitwise_or(frame, engine.img_canvas)
        
        engine.draw_ui(frame)
        engine.update_feedback(frame)
        
        cTime = time.time()
        fps = 1 / (cTime - pTime) if (cTime - pTime) > 0 else 30
        pTime = cTime
        cv2.putText(frame, f'FPS: {int(fps)}', (Config.WIDTH - 80, Config.HEIGHT - 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 100, 100), 1)

        cv2.imshow(Config.APP_NAME, frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2025-12-29 19:32:07.687 python[3627:194014] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
I0000 00:00:1767036728.756307  194014 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1767036728.768873  194322 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767036728.775217  194322 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767036732.546947  194325 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
